## Circuit Data transform Bronze to Silver

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/03-Silver_Helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as f

In [0]:
# To use reader api options 
circuits_df = spark.read.table(bronze_table).filter((f.col("batch_id") == v_batch_id))

# circuits_df = spark.table(bronze_table) - without any reader options
# Both returns the same results

In [0]:
circuits_selected_df = circuits_df.select(
    f.col("circuitId"),
    f.col("circuitName"),
    f.col("lat"),
    f.col("long"),
    f.col("locality"),
    f.col("country"),
    f.col("ingestion_timestamp"),
    f.col("source_file"),
    f.col("batch_id")
)

In [0]:
# For column rename we can use ALIAS funcrion or also the below function

column_rename = {
    "circuitId" : "circuit_id",
    "circuitName" : "circuit_name",
    "lat" : "latitude",
    "long" : "longitude"
    }

circuit_renamed_df = circuits_selected_df.withColumnsRenamed(column_rename)

In [0]:
# To Drop duplicates and Null values in a column - distinct(), dropDuplicates() & filter()

# circuits_valid_df = circuit_renamed_df.distinct().filter(f.col("circuit_id").isNotNull())

circuits_valid_df = circuit_renamed_df.dropDuplicates(["circuit_id"]).filter(f.col("circuit_id").isNotNull())

In [0]:
circuits_final_df = circuits_valid_df.withColumn("circuit_name", f.initcap("circuit_name"))\
                                     . withColumn("locality", f.initcap("locality"))

In [0]:
# circuits_final_df = circuits_final_df.withColumn("created_timestamp", f.current_timestamp())\
#                                       .withColumn("updated_timestamp", f.current_timestamp())

# if not spark.catalog.tableExists(silver_table):

#     circuits_final_df.write.format('delta').mode('overwrite').saveAsTable(silver_table)

# else:

#     from delta.tables import DeltaTable

#     delta_table = DeltaTable.forName(spark, silver_table)

#     delta_table.alias('trg').merge(circuits_final_df.alias('src'), "trg.circuit_id = src.circuit_id")\
#                             .whenMatchedUpdate(
#                                 condition = "src.batch_id >= trg.batch_id",
#                                 set = {
#                                     "circuit_name" : "src.circuit_name",
#                                     "latitude" : "src.latitude",
#                                     "longitude" : "src.longitude",
#                                     "locality" : "src.locality",
#                                     "country" : "src.country",
#                                     "ingestion_timestamp" : "src.ingestion_timestamp",
#                                     "source_file" : "src.source_file",
#                                     "batch_id" : "src.batch_id",
#                                     "updated_timestamp" : "src.updated_timestamp"
#                                 }
#                             ).whenNotMatchedInsertAll().execute()

In [0]:
columns = ["circuit_name",
"latitude",
"longitude",
"locality",
"country",
"ingestion_timestamp",
"source_file",
"batch_id"]

write_to_silver(circuits_final_df, silver_table, "trg.circuit_id = src.circuit_id", columns)